In [0]:
%python
from pyspark.sql.functions import expr,col,current_timestamp,current_date ,lit,sum , max , coalesce,row_number
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType

class OrderProcessor:
    def __init__(self):

        self.spark = spark
        self.checkpointlocati**'

        self.process_dt = datetime.now()

    def read_staging_df(self) -> DataFrame:

        return (
            self.spark.readStream.table("dev.spark_db.order_stg")
        )

    def transaction_processor(self,stg_df:DataFrame) -> DataFrame:

        account = self.spark.read.table("dev.spark_db.account")
        asset = self.spark.read.table("dev.spark_db.asset_rate")

        txn_df = (
            stg_df.alias("sd").join(account.alias("ac"), expr("sd.account_number = ac.account_number"), "inner")
                              .join(asset.alias("as") , 
                                    expr("""
                                         sd.asset_external_id = as.asset_external_id
                                         and 
                                         sd.as_of_dt = as.as_of_dt
                                         """),
                                    "inner")
                              .select("ac.account_number",
                                      "as.asset_external_id",
                                      "sd.order_type",
                                          expr("""
                                           case when sd.order_type = 'BUY' then
                                                    sd.quantity
                                                when sd.order_type = 'SELL' then
                                                    -(sd.quantity)
                                                end
                                           """)
                                      .cast(DecimalType(18,2)).alias("quantity"),
                                      col("sd.asset_rate").cast(DecimalType(18,2)),
                                          expr("""
                                           case when sd.order_type = 'BUY' then
                                                    sd.order_amt
                                                when sd.order_type = 'SELL' then
                                                    -(sd.order_amt)
                                                end
                                           """)
                                      .cast(DecimalType(18,2)).alias("order_amt"),
                                      col("sd.as_of_dt").alias("as_of_dt"),
                                      expr(" 1 as transaction_ver"),
                                      current_timestamp().alias("transaction_dml")
                                      )
        )
        return txn_df
    
    def write_to_transaction(self,stg_df):
        return (
            stg_df.writeStream.format("delta")
                              .queryName("Transaction_Query")
                              .outputMode("append")
                              .option("checkpointLocation" ,self.checkpointlocation_txn)
                              .trigger( availableNow = True)
                              .toTable("dev.spark_db.transaction")
        )

    def main(self):

        read_stg = self.read_staging_df()

        txn = self.transaction_processor(read_stg)

        transaction_query = self.write_to_transaction(txn)

        return transaction_query



if __name__ == '__main__':
    op = OrderProcessor()
    op.main()
                            


